In [1]:
!nvidia-smi

Mon Jul  6 10:34:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!pip install -q virtualenv
!virtualenv -q -p python3.12 /content/vllm_env
!/content/vllm_env/bin/pip install -q --upgrade pip
!/content/vllm_env/bin/pip install -q vllm==0.24.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 48.1 MB/s eta 0:00:00


In [4]:
!/content/vllm_env/bin/python -c "import vllm; print(vllm.__version__)"

0.24.0


In [5]:
%%writefile /content/test_stack.py
from vllm import LLM, SamplingParams

llm = LLM(
    model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    quantization="awq_marlin",
    kv_cache_dtype="fp8",
    speculative_config={
        "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B",
        "method": "eagle3",
        "num_speculative_tokens": 5,
    },
    gpu_memory_utilization=0.85,
)

prompts = ["Q: What is 15% of 240? A:"] * 10
outputs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=64))
for o in outputs:
    print(o.outputs[0].text)

Writing /content/test_stack.py


In [7]:
!/content/vllm_env/bin/pip install -q ninja
!/content/vllm_env/bin/python /content/test_stack.py

INFO 07-06 10:46:09 [api_utils.py:273] non-default args: {'kv_cache_dtype': 'fp8', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'speculative_config': {'model': 'yuhuili/EAGLE3-LLaMA3.1-Instruct-8B', 'method': 'eagle3', 'num_speculative_tokens': 5}, 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'}
INFO 07-06 10:46:11 [model.py:598] Resolved architecture: LlamaForCausalLM
INFO 07-06 10:46:11 [model.py:1725] Using max model len 131072
INFO 07-06 10:46:11 [cache.py:279] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 07-06 10:46:13 [model.py:598] Resolved architecture: LlamaForCausalLM
INFO 07-06 10:46:13 [model.py:1725] Using max model len 2048
INFO 07-06 10:46:13 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.

INFO 07-06 10:46:15 [vllm.py:1006] Asynchronous scheduling is

In [8]:
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python /content/test_stack.py

INFO 07-06 10:48:11 [api_utils.py:273] non-default args: {'kv_cache_dtype': 'fp8', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'speculative_config': {'model': 'yuhuili/EAGLE3-LLaMA3.1-Instruct-8B', 'method': 'eagle3', 'num_speculative_tokens': 5}, 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'}
INFO 07-06 10:48:13 [model.py:598] Resolved architecture: LlamaForCausalLM
INFO 07-06 10:48:13 [model.py:1725] Using max model len 131072
INFO 07-06 10:48:13 [cache.py:279] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor
INFO 07-06 10:48:15 [model.py:598] Resolved architecture: LlamaForCausalLM
INFO 07-06 10:48:15 [model.py:1725] Using max model len 2048
INFO 07-06 10:48:15 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.

INFO 07-06 10:48:17 [vllm.py:1006] Asynchronous scheduling is

In [2]:
import sys
!{sys.executable} -m pip install -q vllm==0.10.1
!{sys.executable} -m pip install -q "transformers>=4.55,<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 414.4/414.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9

In [1]:
import torch, transformers, vllm
print("torch:", torch.__version__, torch.version.cuda, torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("vllm:", vllm.__version__)

2.7.1+cu126 12.6 True
0.10.1


In [2]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    quantization="awq_marlin",
    speculative_config={
        "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B",
        "method": "eagle3",
        "num_speculative_tokens": 5,
    },
    gpu_memory_utilization=0.85,
)

prompts = ["Q: What is 15% of 240? A:"] * 10
outputs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=64))
for o in outputs:
    print(o.outputs[0].text)

INFO 07-06 10:11:03 [__init__.py:241] Automatically detected platform cuda.
INFO 07-06 10:11:05 [utils.py:326] non-default args: {'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4', 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'speculative_config': {'model': 'yuhuili/EAGLE3-LLaMA3.1-Instruct-8B', 'method': 'eagle3', 'num_speculative_tokens': 5}}
INFO 07-06 10:11:22 [__init__.py:711] Resolved architecture: LlamaForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 07-06 10:11:22 [__init__.py:1750] Using max model len 131072
INFO 07-06 10:11:24 [awq_marlin.py:117] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 07-06 10:11:26 [__init__.py:711] Resolved architecture: LlamaForCausalLM
INFO 07-06 10:11:26 [__init__.py:1750] Using max model len 2048
INFO 07-06 10:11:26 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

WARNING 07-06 10:11:29 [__init__.py:2921] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 07-06 10:13:40 [llm.py:298] Supported_tasks: ['generate']


Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 45 D: 48 E: 50
Explanation: We have to find 15% of 240. 15% of 240 = (15/100) × 240 = 36.
Hence, the correct option is (A). 36.
 36 B: 40 C: 

In [3]:
import sys
!{sys.executable} -m pip install -q -U vllm

from vllm import LLM, SamplingParams

llm = LLM(
    model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    quantization="awq_marlin",
    kv_cache_dtype="fp8",
    speculative_config={
        "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B",
        "method": "eagle3",
        "num_speculative_tokens": 5,
    },
    gpu_memory_utilization=0.85,
)

prompts = ["Q: What is 15% of 240? A:"] * 10
outputs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=64))
for o in outputs:
    print(o.outputs[0].text)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.2/279.2 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748.7 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53

ModuleNotFoundError: No module named 'vllm.attention.utils.fa_utils'